In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
train_df = pd.read_csv("train_df.csv")


In [ ]:
train_df.drop_duplicates(inplace=True)

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4996 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   age                4996 non-null   int64 
 1   gender             4996 non-null   object
 2   primary_diagnosis  4996 non-null   object
 3   num_procedures     4996 non-null   int64 
 4   days_in_hospital   4996 non-null   int64 
 5   comorbidity_score  4996 non-null   int64 
 6   discharge_to       4996 non-null   object
 7   readmitted         4996 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 351.3+ KB


In [ ]:
train_df.head()

,age,gender,primary_diagnosis,num_procedures,days_in_hospital,comorbidity_score,discharge_to,readmitted
0,69,Male,Heart Disease,1,2,1,Home Health Care,0
1,32,Female,COPD,2,13,2,Rehabilitation Facility,0
2,89,Male,Diabetes,1,7,1,Home,0
3,78,Male,COPD,9,2,2,Skilled Nursing Facility,0
4,38,Male,Diabetes,6,4,4,Rehabilitation Facility,0


In [ ]:
x = train_df.drop('readmitted', axis = 1)
y = train_df['readmitted']

In [ ]:
nums = ["age", 'num_procedures','days_in_hospital','comorbidity_score' ]
cat_cols = ['gender', 'primary_diagnosis','discharge_to']


In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), nums),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_cols),
    ]
)

In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, random_state=42)),
    #("classifier", HistGradientBoostingClassifier(random_state=42, class_weight="balanced")),
])

In [ ]:
pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'num_procedures',
                                                   'days_in_hospital',
                                                   'comorbidity_score']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['gender',
                                                   'primary_diagnosis',
                                                   'discharge_to'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [ ]:
val_probs = pipeline.predict_proba(x_val)[:, 1]

In [ ]:
val_acc = roc_auc_score(y_val, val_probs)

In [ ]:
print(f"ROC Score: ", val_acc)

ROC Score:  0.49221046172725874


In [ ]:
pipeline.fit(x, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'num_procedures',
                                                   'days_in_hospital',
                                                   'comorbidity_score']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['gender',
                                                   'primary_diagnosis',
                                                   'discharge_to'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [ ]:
test_df = pd.read_csv("test_df.csv")
sub_df = pd.read_csv("sample_submission.csv")

In [ ]:
sub_df["readmitted"] = pipeline.predict_proba(test_df)[:, 1]

In [ ]:
sub_df.to_csv("submission.csv", index=False)